In [ ]:
"""
SHAP_All_Models.py
==================
Loads SHAP artifacts from Feature_Selected_Retrain.py output dir and runs
SHAP analysis for all 4 models (DT, Lasso, XGB, Tuned RF) on the last fold.

Produces:
  - beeswarm plots (feature directionality)
  - bar plots (mean |SHAP|)
  - per-model SHAP value CSVs
  - combined top-feature comparison across models
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import pickle
import os
import warnings
from scipy.sparse import load_npz

warnings.filterwarnings('ignore')

In [ ]:
# =============================================================
# CONFIG
# =============================================================
ARTIFACT_DIR = "shap_artifacts_feature_selected"
FOLD_IDX = 4                # last fold (0-indexed) — change if you want a different fold
TOP_N_SHAP = 30             # how many features to show in plots
OUTPUT_DIR = "shap_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# =============================================================
# LOAD SHARED ARTIFACTS
# =============================================================
print("=" * 60)
print("LOADING ARTIFACTS")
print("=" * 60)

feature_cols = np.load(os.path.join(ARTIFACT_DIR, "feature_cols.npy"), allow_pickle=True)
feature_cols = list(feature_cols)
print(f"Features: {len(feature_cols)}")

X_test_sparse = load_npz(os.path.join(ARTIFACT_DIR, f"X_test_fold{FOLD_IDX}.npz"))
y_test = np.load(os.path.join(ARTIFACT_DIR, f"y_test_fold{FOLD_IDX}.npy"))

# SHAP needs dense for most explainers
X_test = X_test_sparse.toarray().astype(np.float32)
print(f"X_test shape: {X_test.shape}")
print(f"y_test: {len(y_test)} samples (pos={y_test.sum()}, neg={len(y_test)-y_test.sum()})")

# wrap in DataFrame for nice feature names in plots
X_test_df = pd.DataFrame(X_test, columns=feature_cols)

In [ ]:
# =============================================================
# LOAD MODELS
# =============================================================
print("\nLoading models...")

with open(os.path.join(ARTIFACT_DIR, f"dt_fold{FOLD_IDX}.pkl"), "rb") as f:
    dt_model = pickle.load(f)
print("  Decision Tree: loaded")

with open(os.path.join(ARTIFACT_DIR, f"lasso_fold{FOLD_IDX}.pkl"), "rb") as f:
    lasso_model, lasso_scaler = pickle.load(f)
print("  Lasso: loaded (model + scaler)")

with open(os.path.join(ARTIFACT_DIR, f"xgb_fold{FOLD_IDX}.pkl"), "rb") as f:
    xgb_model = pickle.load(f)
print("  XGBoost: loaded")

with open(os.path.join(ARTIFACT_DIR, f"rf_tuned_fold{FOLD_IDX}.pkl"), "rb") as f:
    rf_model = pickle.load(f)
print("  Tuned RF: loaded")

In [ ]:
# =============================================================
# HELPER: extract SHAP values for class 1 (disease)
# =============================================================
def extract_shap_values_class1(shap_values_raw, n_samples, n_features):
    """
    Handle SHAP version differences. Newer SHAP returns a 3D array
    (n_samples, n_features, n_classes) for tree models, older versions
    return a list of 2 arrays. We want class 1 (disease).
    """
    if isinstance(shap_values_raw, list):
        # older shap: list of [class0_array, class1_array]
        return shap_values_raw[1]
    elif hasattr(shap_values_raw, 'values'):
        # shap.Explanation object
        vals = shap_values_raw.values
        if vals.ndim == 3:
            return vals[:, :, 1]
        return vals
    elif isinstance(shap_values_raw, np.ndarray):
        if shap_values_raw.ndim == 3:
            return shap_values_raw[:, :, 1]
        return shap_values_raw
    else:
        raise ValueError(f"unexpected shap_values type: {type(shap_values_raw)}")

In [ ]:
# =============================================================
# DECISION TREE — SHAP
# =============================================================
print("\n" + "=" * 60)
print("SHAP: DECISION TREE")
print("=" * 60)

dt_explainer = shap.TreeExplainer(dt_model)
dt_shap_raw = dt_explainer.shap_values(X_test)
dt_shap = extract_shap_values_class1(dt_shap_raw, X_test.shape[0], X_test.shape[1])

print(f"SHAP values shape: {dt_shap.shape}")

# beeswarm
fig, ax = plt.subplots(figsize=(12, 10))
shap_explanation = shap.Explanation(
    values=dt_shap,
    base_values=np.full(dt_shap.shape[0],
                        dt_explainer.expected_value[1] if isinstance(dt_explainer.expected_value, (list, np.ndarray))
                        else dt_explainer.expected_value),
    data=X_test,
    feature_names=feature_cols
)
shap.plots.beeswarm(shap_explanation, max_display=TOP_N_SHAP, show=False)
plt.title("SHAP Beeswarm — Decision Tree (class 1: Disease)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "DT_shap_beeswarm.png"), dpi=150, bbox_inches='tight')
plt.close()

# bar
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.bar(shap_explanation, max_display=TOP_N_SHAP, show=False)
plt.title("SHAP Bar — Decision Tree (class 1: Disease)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "DT_shap_bar.png"), dpi=150, bbox_inches='tight')
plt.close()

print("Saved: DT_shap_beeswarm.png, DT_shap_bar.png")

In [ ]:
# =============================================================
# LASSO — SHAP
# =============================================================
print("\n" + "=" * 60)
print("SHAP: LASSO")
print("=" * 60)

# scale X_test with the fold's scaler before explaining
X_test_scaled = lasso_scaler.transform(X_test).astype(np.float32)

# LinearExplainer for logistic regression
lasso_explainer = shap.LinearExplainer(lasso_model, X_test_scaled)
lasso_shap_raw = lasso_explainer.shap_values(X_test_scaled)

# LinearExplainer returns 2D for binary classification
if isinstance(lasso_shap_raw, list):
    lasso_shap = lasso_shap_raw[1]
elif isinstance(lasso_shap_raw, np.ndarray) and lasso_shap_raw.ndim == 3:
    lasso_shap = lasso_shap_raw[:, :, 1]
else:
    lasso_shap = lasso_shap_raw

print(f"SHAP values shape: {lasso_shap.shape}")

# NOTE: for beeswarm, use UNSCALED feature values so the color axis
# shows actual clinical values, not z-scores
lasso_explanation = shap.Explanation(
    values=lasso_shap,
    base_values=np.full(lasso_shap.shape[0],
                        lasso_explainer.expected_value[1] if isinstance(lasso_explainer.expected_value, (list, np.ndarray))
                        else lasso_explainer.expected_value),
    data=X_test,  # unscaled for interpretability
    feature_names=feature_cols
)

fig, ax = plt.subplots(figsize=(12, 10))
shap.plots.beeswarm(lasso_explanation, max_display=TOP_N_SHAP, show=False)
plt.title("SHAP Beeswarm — Lasso (class 1: Disease)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Lasso_shap_beeswarm.png"), dpi=150, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.bar(lasso_explanation, max_display=TOP_N_SHAP, show=False)
plt.title("SHAP Bar — Lasso (class 1: Disease)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Lasso_shap_bar.png"), dpi=150, bbox_inches='tight')
plt.close()

print("Saved: Lasso_shap_beeswarm.png, Lasso_shap_bar.png")

del X_test_scaled

In [ ]:
# =============================================================
# XGBOOST — SHAP
# =============================================================
print("\n" + "=" * 60)
print("SHAP: XGBOOST")
print("=" * 60)

xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap_raw = xgb_explainer.shap_values(X_test)
xgb_shap = extract_shap_values_class1(xgb_shap_raw, X_test.shape[0], X_test.shape[1])

# XGBoost TreeExplainer sometimes returns single-output for binary;
# if so, positive SHAP = pushes toward class 1 already
if isinstance(xgb_shap_raw, np.ndarray) and xgb_shap_raw.ndim == 2:
    xgb_shap = xgb_shap_raw  # already class-1 log-odds

print(f"SHAP values shape: {xgb_shap.shape}")

xgb_expected = xgb_explainer.expected_value
if isinstance(xgb_expected, (list, np.ndarray)):
    xgb_base = xgb_expected[1] if len(xgb_expected) > 1 else xgb_expected[0]
else:
    xgb_base = xgb_expected

xgb_explanation = shap.Explanation(
    values=xgb_shap,
    base_values=np.full(xgb_shap.shape[0], xgb_base),
    data=X_test,
    feature_names=feature_cols
)

fig, ax = plt.subplots(figsize=(12, 10))
shap.plots.beeswarm(xgb_explanation, max_display=TOP_N_SHAP, show=False)
plt.title("SHAP Beeswarm — XGBoost (class 1: Disease)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "XGB_shap_beeswarm.png"), dpi=150, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.bar(xgb_explanation, max_display=TOP_N_SHAP, show=False)
plt.title("SHAP Bar — XGBoost (class 1: Disease)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "XGB_shap_bar.png"), dpi=150, bbox_inches='tight')
plt.close()

print("Saved: XGB_shap_beeswarm.png, XGB_shap_bar.png")

In [ ]:
# =============================================================
# TUNED RANDOM FOREST — SHAP
# =============================================================
print("\n" + "=" * 60)
print("SHAP: TUNED RANDOM FOREST")
print("=" * 60)

# interventional feature perturbation for RF (avoids correlated-feature artifacts)
rf_explainer = shap.TreeExplainer(
    rf_model,
    feature_perturbation="interventional",
    data=shap.sample(X_test, min(200, X_test.shape[0]))
)
rf_shap_raw = rf_explainer.shap_values(X_test)
rf_shap = extract_shap_values_class1(rf_shap_raw, X_test.shape[0], X_test.shape[1])

print(f"SHAP values shape: {rf_shap.shape}")

rf_expected = rf_explainer.expected_value
if isinstance(rf_expected, (list, np.ndarray)):
    rf_base = rf_expected[1] if len(rf_expected) > 1 else rf_expected[0]
else:
    rf_base = rf_expected

rf_explanation = shap.Explanation(
    values=rf_shap,
    base_values=np.full(rf_shap.shape[0], rf_base),
    data=X_test,
    feature_names=feature_cols
)

fig, ax = plt.subplots(figsize=(12, 10))
shap.plots.beeswarm(rf_explanation, max_display=TOP_N_SHAP, show=False)
plt.title("SHAP Beeswarm — Tuned RF (class 1: Disease)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "RF_shap_beeswarm.png"), dpi=150, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.bar(rf_explanation, max_display=TOP_N_SHAP, show=False)
plt.title("SHAP Bar — Tuned RF (class 1: Disease)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "RF_shap_bar.png"), dpi=150, bbox_inches='tight')
plt.close()

print("Saved: RF_shap_beeswarm.png, RF_shap_bar.png")

In [ ]:
# =============================================================
# SAVE SHAP VALUES AS CSV
# =============================================================
print("\n" + "=" * 60)
print("SAVING SHAP VALUE MATRICES")
print("=" * 60)

for name, sv in [('DT', dt_shap), ('Lasso', lasso_shap),
                 ('XGB', xgb_shap), ('RF', rf_shap)]:
    sv_df = pd.DataFrame(sv, columns=feature_cols)
    sv_df.to_csv(os.path.join(OUTPUT_DIR, f"{name}_shap_values.csv"), index=False)
    print(f"  {name}_shap_values.csv: {sv_df.shape}")

In [ ]:
# =============================================================
# CROSS-MODEL COMPARISON: TOP FEATURES BY MEAN |SHAP|
# =============================================================
print("\n" + "=" * 60)
print("CROSS-MODEL SHAP COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({'feature': feature_cols})

for name, sv in [('DT', dt_shap), ('Lasso', lasso_shap),
                 ('XGB', xgb_shap), ('RF', rf_shap)]:
    mean_abs = np.mean(np.abs(sv), axis=0)
    # normalize to [0, 1] within each model for comparability
    if mean_abs.max() > 0:
        mean_abs_norm = mean_abs / mean_abs.max()
    else:
        mean_abs_norm = mean_abs
    comparison[f'{name}_mean_abs_shap'] = mean_abs
    comparison[f'{name}_norm'] = mean_abs_norm
    comparison[f'{name}_rank'] = mean_abs.argsort().argsort() + 1  # 1-based rank

# average normalized SHAP across models
norm_cols = [c for c in comparison.columns if c.endswith('_norm')]
comparison['avg_norm_shap'] = comparison[norm_cols].mean(axis=1)
comparison = comparison.sort_values('avg_norm_shap', ascending=False)

comparison.to_csv(os.path.join(OUTPUT_DIR, "cross_model_shap_comparison.csv"), index=False)

# print top features
print(f"\nTop {TOP_N_SHAP} features by average normalized |SHAP| across all models:\n")
print(f"{'Feature':45s} {'DT rank':>8s} {'Lasso':>8s} {'XGB':>8s} {'RF':>8s} {'Avg norm':>9s}")
print("-" * 90)
for _, row in comparison.head(TOP_N_SHAP).iterrows():
    n = len(feature_cols)
    print(f"{row['feature']:45s} "
          f"{int(n - row['DT_rank'] + 1):>8d} "
          f"{int(n - row['Lasso_rank'] + 1):>8d} "
          f"{int(n - row['XGB_rank'] + 1):>8d} "
          f"{int(n - row['RF_rank'] + 1):>8d} "
          f"{row['avg_norm_shap']:>9.4f}")

In [ ]:
# =============================================================
# COMBINED BAR CHART — TOP FEATURES ACROSS MODELS
# =============================================================
top_feats = comparison.head(TOP_N_SHAP)['feature'].tolist()

fig, ax = plt.subplots(figsize=(14, 10))
y_pos = np.arange(len(top_feats))
bar_width = 0.2

for j, (name, color) in enumerate([
    ('DT', '#2196F3'), ('Lasso', '#FF5722'),
    ('XGB', '#9C27B0'), ('RF', '#4CAF50')
]):
    vals = comparison.set_index('feature').loc[top_feats, f'{name}_norm'].values
    ax.barh(y_pos + j * bar_width, vals[::-1], bar_width,
            label=name, color=color, alpha=0.85)

ax.set_yticks(y_pos + 1.5 * bar_width)
ax.set_yticklabels(top_feats[::-1], fontsize=8)
ax.set_xlabel('Normalized mean |SHAP|')
ax.set_title(f'Top {TOP_N_SHAP} Features — Cross-Model SHAP Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cross_model_shap_comparison.png"), dpi=150, bbox_inches='tight')
plt.close()
print(f"\nSaved: cross_model_shap_comparison.png")

In [ ]:
# =============================================================
# SUMMARY
# =============================================================
print("\n" + "=" * 60)
print("ALL OUTPUTS")
print("=" * 60)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fsize = os.path.getsize(os.path.join(OUTPUT_DIR, fname))
    unit = "KB" if fsize < 1024 * 1024 else "MB"
    val = fsize / 1024 if unit == "KB" else fsize / (1024 * 1024)
    print(f"  {fname:50s} ({val:.1f} {unit})")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)